# 07 - Railway Signalling Spare Parts: Inventory Analysis

**Southern Railway S&T - Strategic & Operational Inventory Analytics Platform**

This notebook is an **orchestration + explanation** layer. It contains *no business logic* -
every calculation lives in the `railway/` package modules. Cells either call module build
functions with `write=False` (compute without overwriting saved outputs) or read the
already-generated CSVs.

## 1. Introduction
Two independent pipelines (never merged):
- **Strategic** (`raw_data/railways.xlsx`, 59 vital items) - forecasting, ABC, criticality, optimization, AnyLogistix.
- **Operational** (`raw_data/railway_stock_summary.xlsx`, 907 depot items) - aging, dead stock, slow-moving, valuation.

The Walmart workflow (notebooks 01-06) is completely untouched.


## 2. Architecture Overview

```mermaid
flowchart TD
  R1[raw_data/railways.xlsx] --> SP[Strategic Pipeline]
  R2[raw_data/railway_stock_summary.xlsx] --> OP[Operational Pipeline]
  SP --> DH[demand_history] --> SM[sku_master] --> FC[forecast] --> POL[inventory_policy]
  POL --> DQ[Data Quality Layer]
  SM --> EX[Executive Layer]
  FC --> EX
  POL --> EX
  DQ --> EX
  OP --> OPI[operational_inventory]
  SM --> RAT[Rationalization]
  POL --> RAT
  OPI --> RAT
  EX --> PBI[Power BI page0-9]
  POL --> PBI
  OPI --> PBI
  RAT --> PBI
  DQ --> PBI
  SM --> ALX[AnyLogistix Digital Twin]
  FC --> ALX
  POL --> ALX
  OPI --> ALX
  RAT --> ALX
```


In [1]:
import sys
from pathlib import Path
import pandas as pd
ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
sys.path.insert(0, str(ROOT))
from railway import (railway_data_preparation as dp, railway_classification as rc,
                     railway_forecasting as rf, railway_inventory_optimization as opt,
                     railway_operational_analysis as op, railway_inventory_rationalization as rat,
                     railway_config as cfg, schema_validation as sv, railway_regression as reg)
pd.set_option('display.max_columns', 30); pd.set_option('display.width', 200)
print('Railway modules loaded from', ROOT)


Railway modules loaded from C:\Users\uppuk\SparePartsInventory


## 3. Strategic Pipeline
Demand history -> SKU master (ABC via AAC x Unit_Cost, criticality S/V/N -> S1/S2/S3, demand class, coverage).


In [2]:
demand = dp.build_demand_history(write=False)
master = rc.build_sku_master(write=False)
print('Demand history:', demand.shape, '| SKU master:', master.shape)
print('ABC:', master['ABC_Class'].value_counts().reindex(cfg.ABC_ORDER).to_dict())
print('Criticality:', master['Criticality'].value_counts().reindex(cfg.CRITICALITY_ORDER).to_dict())
print(master[['PL_Code','Description','ABC_Class','Criticality','Demand_Class','Coverage_Class']].head().to_string(index=False))


Demand history: (59, 12) | SKU master: (59, 19)
ABC: {'A': 9, 'B1': 4, 'B2': 6, 'C': 40}
Criticality: {'S1': 13, 'S2': 14, 'S3': 17, 'S4': 15}
 PL_Code                                       Description ABC_Class Criticality Demand_Class    Coverage_Class
50540324  Lead Acid Stationary Secondary Cells  - 2V/80 AH         A          S2       Smooth Critical Shortage
56501018                            Cable 30 C x 1.5 Sq.mm         A          S1       Smooth      Understocked
50550081 Lead Acid Stationary Secondary Cells  - 2V/200 AH         A          S2       Smooth Critical Shortage
56509959                            Cable 12 C x 1.5 Sq.mm         A          S1      Erratic      Understocked
56468039                   LED Red aspect with accessories         A          S1       Smooth      Understocked


## 4. Operational Pipeline
`railway_stock_summary.xlsx` only - aging, movement, value bands, Pareto ABC.


In [3]:
operational = op.build_operational_inventory(write=False)
print('Operational rows:', len(operational))
print('Aging:', operational['Inventory_Aging_Class'].value_counts().to_dict())
print('Movement:', operational['Movement_Status'].value_counts().to_dict())


Operational rows: 907
Aging: {'Very Slow Moving': 232, 'Active': 202, 'Dead Stock': 190, 'Monitor': 142, 'Slow Moving': 95, 'Unknown': 46}
Movement: {'Active': 344, 'Slow Moving': 327, 'Dead Stock': 190, 'Unknown': 46}


## 5. Forecasting Results
Primary blend (0.40 AAC + 0.30 EAR + 0.20 MA + 0.10 CAGR) + Croston/SBA/TSB/Holt benchmark + 4->1 backtest.


In [4]:
forecast, _ = rf.build_forecast(write=False)
print('Recommended model:', forecast['Recommended_Model'].value_counts().to_dict())
print('Confidence:', forecast['Forecast_Confidence'].value_counts().to_dict())
print(forecast[['PL_Code','Forecast_2026_27','Recommended_Model','Forecast_Confidence']].head().to_string(index=False))


Recommended model: {'Weighted Blend': 30, 'Zero Forecast': 14, 'Croston': 7, 'SBA': 7, 'TSB': 1}
Confidence: {'High': 22, 'Medium': 15, 'Very Low': 14, 'Low': 8}
          PL_Code  Forecast_2026_27 Recommended_Model Forecast_Confidence
         56509376          14956.92    Weighted Blend                High
         56509388           5315.06    Weighted Blend                High
         67900203             10.50     Zero Forecast            Very Low
         56500350             28.72           Croston              Medium
52156576/56500452             45.37           Croston              Medium


## 6. Inventory Optimization Results
Tier-1/Tier-2 lead time -> safety stock, ROP, min/max, criticality-weighted priority.


In [5]:
policy = opt.build_inventory_policy(write=False)
print('Inventory status:', policy['Inventory_Status'].value_counts().to_dict())
print('Priority class:', policy['Procurement_Priority_Class'].value_counts().to_dict())
print(policy[['PL_Code','Criticality','Lead_Time_Months','Safety_Stock','ROP','Inventory_Status']].head().to_string(index=False))


Inventory status: {'Procurement Required': 50, 'Sufficient': 9}
Priority class: {'Low': 23, 'Medium': 18, 'High': 12, 'Immediate': 6}
          PL_Code Criticality  Lead_Time_Months  Safety_Stock      ROP     Inventory_Status
         56119033          S1              12.0       5435.95 15650.23 Procurement Required
56987122/56110029          S1              12.0       8779.92 11057.48 Procurement Required
         50550081          S2              12.0      11977.97 16113.33 Procurement Required
         56509960          S1              12.0      17437.14 39069.08 Procurement Required
         56501018          S1               6.0      58424.33 72606.45 Procurement Required


## 7. Rationalization Results
Unified asset register -> Procure / Retain / Monitor / Rationalize / Dispose.


In [6]:
rationalization = rat.build_rationalization(write=False)
print('Actions:', rationalization['Inventory_Action'].value_counts().to_dict())
print(rationalization.groupby('Inventory_Action')['Inventory_Value'].sum().round(0).to_dict())


Actions: {'Retain': 348, 'Rationalize': 327, 'Dispose': 190, 'Monitor': 71, 'Procure Immediately': 23}
{'Dispose': 76283548.0, 'Monitor': 7104688.0, 'Procure Immediately': 76978850.0, 'Rationalize': 164178536.0, 'Retain': 270750124.0}


## 8. Executive Dashboard Results
Read the generated KPI summary (no recomputation).


In [7]:
ek = pd.read_csv(cfg.OUTPUT_DIR / 'executive_kpi_summary.csv')
print(ek[ek['Section'] == 'Section_Management_Insights'][['KPI','Value']].to_string(index=False))
print()
print(ek[ek['KPI'].isin(['Total_Inventory_Value','Procurement_Required_Items','Data_Quality_Impact'])].to_string(index=False))


                 KPI                                                                         Value
Management_Insight_1                      50 of 59 strategic signalling items require procurement.
Management_Insight_2 S1 (track-safety) items account for 55.0% of normalized procurement exposure.
Management_Insight_3     Only 2 data-quality anomalies inflated investment estimates by Rs 613 Cr.
Management_Insight_4             Top 10 SKUs account for 82% of (unit-normalized) inventory value.

                              Section                        KPI         Value
Section1_Strategic_Inventory_Overview      Total_Inventory_Value  662075255.38
        Section3_Procurement_Exposure Procurement_Required_Items            50
                Section5_Data_Quality        Data_Quality_Impact 6133228353.09


## 9. Power BI Outputs
Unified semantic layer (visualization-only): 10 pages + 5 operational aggregates.


In [8]:
pages = sorted(p.name for p in cfg.POWERBI_DIR.glob('page*.csv'))
ops = sorted(p.name for p in cfg.POWERBI_DIR.glob('op_*.csv'))
print('Power BI pages :', pages)
print('Operational agg:', ops)
print(pd.read_csv(cfg.POWERBI_DIR / 'page0_executive_dashboard.csv').to_string(index=False))


Power BI pages : ['page0_executive_dashboard.csv', 'page1_procurement.csv', 'page2_forecasting.csv', 'page3_criticality.csv', 'page4_operational_health.csv', 'page5_rationalization.csv', 'page6_data_quality.csv', 'page7_abc_criticality_matrix.csv', 'page8_budget_scenarios.csv', 'page9_management_actions.csv']
Operational agg: ['op_dead_stock.csv', 'op_inventory_aging.csv', 'op_inventory_summary.csv', 'op_inventory_value.csv', 'op_operational_abc.csv']
                                   KPI        Value    Universe
Strategic Inventory Value (Normalized) 8.566364e+07   Strategic
           Operational Inventory Value 5.124064e+08 Operational
                    Annual Issue Value 1.613748e+08   Strategic
            Procurement Required Value 4.731154e+08   Strategic
                      Dead Stock Value 7.628355e+07 Operational
                     Slow Moving Value 1.641785e+08 Operational
                 Inventory Turn Risk % 4.693000e+01 Operational
         Inventory Concentration

## 10. AnyLogistix Outputs
Digital-twin foundation: locations, products, demand, policy, facilities, service risk, multi-echelon, scenarios, readiness.


In [9]:
alx = sorted(p.name for p in cfg.ANYLOGISTIX_DIR.glob('*.csv'))
print('AnyLogistix files:', alx)
print()
print(pd.read_csv(cfg.ANYLOGISTIX_DIR / 'digital_twin_readiness.csv').to_string(index=False))


AnyLogistix files: ['demand.csv', 'digital_twin_readiness.csv', 'facilities.csv', 'inventory_policy.csv', 'locations.csv', 'multi_echelon_candidates.csv', 'procurement_scenarios.csv', 'products.csv', 'service_risk.csv']

                        Metric  Score_Pct       Band
     Network_Data_Completeness       50.0    Partial
   Inventory_Data_Completeness      100.0      Ready
      Demand_Data_Completeness       50.0    Partial
      Policy_Data_Completeness      100.0      Ready
Overall_Digital_Twin_Readiness       75.0 Near Ready


## 11. Production Readiness
Fail-fast schema validation + regression drift guard.


In [10]:
violations = sv.validate_all(raise_on_error=False)
print('Schema validation:', 'ALL CHECKS PASSED' if not violations else violations)
drift = reg.compare_to_baseline()
print('Regression:', 'no drift - matches baseline' if not drift else drift)


Schema validation: ALL CHECKS PASSED
Regression: no drift - matches baseline


## 12. Key Limitations
1. **5 annual observations** per SKU (2021-22 missing) - forecasts are *directional*, not contractual.
2. **Locked Safety_Stock formula** mixes annual sigma with sqrt(months) - magnitudes are conservative; use as *relative ranking*.
3. **2 cable SKUs** carry per-km rates (normalized in the Data Quality layer; originals preserved).
4. **No geo-coordinates / division-level demand** in generated outputs - AnyLogistix demand uses Equal_Split.
5. Strategic (59) and operational (907) sets overlap on only ~7 PL codes - never force-joined.


## 13. Recommended Next Steps
1. Ingest **monthly issue transactions** to lift forecasting beyond 5 annual points.
2. Supply **real division/depot coordinates** and **division consumption %** to enable network modeling.
3. Wire the **Asset_Type** classifier into the strategic master for store-type coverage (Kavach/IPS/relay).
4. Vectorise the per-SKU loops before scaling beyond ~100k items.
5. Establish the **depot -> division** hierarchy for multi-echelon studies.

See `RAILWAY_PRODUCTION_READINESS_REPORT.md` and `README_RAILWAYS.md` for the full assessment.
